In [ ]:
import os
import gc
import cv2
import time
import random
from monai.inferers import sliding_window_inference

# For data manipulation
import numpy as np
import pandas as pd

# Pytorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.cuda import amp

# Utils
from tqdm.auto import tqdm

# For Image Models
import timm
from timm.utils import ModelEmaV2

# Albumentations for augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

import warnings
warnings.filterwarnings("ignore")

## using gpu:1
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

def seed_everything(seed=123):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

In [ ]:
class Customize_Model(nn.Module):
    def __init__(self, model_name, cls):
        super().__init__()
        self.cls= cls
        self.model = timm.create_model(model_name, 
                                       pretrained=True, 
                                       num_classes=0, 
                                       drop_rate= CFG['drop_out'], 
                                       drop_path_rate= CFG['drop_path']
                                    )
        self.linear = nn.Linear(self.model.num_features, cls)
        
    def forward(self, image):
        x = self.model(image)
        x = self.linear(x)
        x = torch.expm1(x) if not self.training else x
        return x
    

from transformers import Dinov2ForImageClassification, Dinov2Config
class huggingface_model(nn.Module):
    def __init__(self, cls):
        super().__init__()
        print("Use HuggingFace DINOv2 classification model")
        self.cls = cls
        self.model = Dinov2ForImageClassification.from_pretrained(
            "facebook/dinov2-small-imagenet1k-1-layer"
        )

        in_dim = self.model.classifier.in_features
        self.model.classifier = nn.Linear(in_dim, cls)

    def forward(self, images):
        out = self.model(images).logits
        return out
    
    
class dinov3(nn.Module):
    def __init__(self, cls):
        super().__init__()
        self.cls_num = cls
        self.model =torch.hub.load("dinov3", 'dinov3_vitl16', source='local', weights="dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")
        # self.model =torch.hub.load("dinov3", 'dinov3_convnext_large', source='local', weights="dinov3_convnext_large_pretrain_lvd1689m-61fa432d.pth")
        feature_dim = self.model.embed_dim * 4  # 因為 cat 了 4 層
        self.single_head = nn.Linear(feature_dim, cls)
        self.multi_head = nn.ModuleList([
            nn.Linear(feature_dim, 1) for _ in range(self.cls_num)
        ])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
    def forward(self, image):
        layers = self.model.get_intermediate_layers(image, n=4, reshape=True)
        pooled_layers = [self.pool(ly) for ly in layers]
        x = torch.cat(pooled_layers, dim=1).squeeze(-1).squeeze(-1)
        out = self.single_head(x)
        # out = torch.cat([head(x) for head in self.heads], dim=1)
        out = torch.expm1(out) if not self.training else out
        return out
    
from surgicaldino import SurgicalDINO
class dinov2_lora(nn.Module):
    def __init__(self, cls):
        super().__init__()
        self.cls_num = cls
        self.model = SurgicalDINO(backbone_size="base", r=192, lora_layer=None, decode_type = 'linear4')
        feature_dim = 768 * 4  # 因為 cat 了 4 層
        self.single_head = nn.Linear(feature_dim, cls)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
    def forward(self, image):
        layers = self.model(pixel_values = image)
        pooled_layers = [self.pool(ly) for ly in layers]
        x = torch.cat(pooled_layers, dim=1).squeeze(-1).squeeze(-1)
        out = self.single_head(x)
        out = torch.expm1(out) if not self.training else out
        return out
    

from surgicaldino import SurgicalDINO
class dinov2v3_fuse(nn.Module):
    def __init__(self, cls):
        super().__init__()
        self.cls_num = cls
        self.dinov2 = SurgicalDINO(backbone_size="base", r=192, lora_layer=None, decode_type = 'linear4')
        self.dinov3 =torch.hub.load("dinov3", 'dinov3_vitl16', source='local', weights="dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")

        feature_dim = self.dinov3.embed_dim * 4 + 768 * 4  # 因為 cat 了 4 層
        self.single_head = nn.Linear(feature_dim, cls)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
    def forward(self, image):
        layers = self.dinov2(pixel_values = image)
        pooled_layers = [self.pool(ly) for ly in layers]
        v2_feats = torch.cat(pooled_layers, dim=1).squeeze(-1).squeeze(-1)

        layers = self.dinov3.get_intermediate_layers(image, n=4, reshape=True)
        pooled_layers = [self.pool(ly) for ly in layers]
        v3_feats = torch.cat(pooled_layers, dim=1).squeeze(-1).squeeze(-1)

        x = torch.cat([v2_feats, v3_feats], dim=1)
        out = self.single_head(x)
        out = torch.expm1(out) if not self.training else out
        return out

    
# x = torch.rand(1,3,224,448)
# model = dinov2v3_fuse(cls=5)
# out = model(x)
# print(out.shape)

In [ ]:
def get_train_transform(img_size):
    return A.Compose([
        # A.Resize(img_size, img_size),
        A.SmallestMaxSize(max_size=img_size, interpolation=3, p=1),
        
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.CLAHE(clip_limit=4.0, p=0.7),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
        A.OneOf([
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=5),
            A.GaussianBlur(blur_limit=5),
            A.GaussNoise(var_limit=(5.0, 30.0)),
        ], p=0.7),
        
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        # A.Transpose(p=0.5),
        # A.OneOf([
        #     A.GridDistortion(num_steps=5, distort_limit=0.05, p=1.0),
        #     A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0)
        # ], p=0.3),
        A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.05, rotate_limit= 10,
                                        interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT, p=0.7),

        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1.0),
    ])


def get_test_transform(img_size):
    return A.Compose([
        A.SmallestMaxSize(max_size=img_size, interpolation=3, p=1),
        # A.Resize(img_size, img_size),
        
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1.0),
    ])

In [ ]:
class Customize_Dataset(Dataset):
    def __init__(self, df, transforms=None, training=False):
        self.df = df
        self.transforms = transforms
        self.training= training
    
    def mixup_aug(self, img_1, mask_1, 
                        img_2, mask_2):
        """
        img: numpy array of shape (height, width,channel)
        mask: numpy array of shape (height, width,channel)
        """
        ## mixup
        weight= 0.5 #np.random.beta(a=0.5, b=0.5)
        img= img_1*weight + img_2*(1-weight)
        mask= mask_1*weight + mask_2*(1-weight)
        return img.astype(np.uint8), mask
    
    def read_data(self, data):
        img = cv2.imread(data['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        label= data.values[4:-1].astype(np.float32)
        label= np.array(label)
        return img, label
    
    def __getitem__(self, index):
        data = self.df.loc[index]
        img, label= self.read_data(data)
        
        # use mixup
        if self.training and np.random.rand() >= (1-CFG['mixup']):
            img_1= img
            label_1= np.array(label)
            while True:
                indx= np.random.randint(len(self.df))
                data= self.df.loc[indx]
                img_2, label_2= self.read_data(data)
                if label_1.argmax(0)!=label_2.argmax(0): break
            img, label= self.mixup_aug(img_1, label_1, 
                                       img_2, label_2)
        
        if self.transforms:
            img = self.transforms(image=img)["image"]
            
        return {
            'image': torch.tensor(img, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.float32),
        }
    
    def __len__(self):
        return len(self.df)

In [ ]:
class Customize_loss(nn.Module):
    def __init__(self):
        super().__init__()
        self.MAE = nn.L1Loss()
        self.MSE = nn.MSELoss()
        self.Huber = nn.HuberLoss(delta=1.0)

    def log_cosh_loss(self, y_pred, y_true):
        # Log-Cosh 公式: log(cosh(x))，在 x 很大時近似於 |x| - log(2)
        # 為了數值穩定性，使用 log(exp(x) + exp(-x)) - log(2) 的變體
        ey = y_pred - y_true
        return torch.mean(torch.log(torch.cosh(ey + 1e-12)))

    def quantile_loss(self, y_pred, y_true, tau=0.5):
        # 分位數損失 (Pinball Loss)
        # tau=0.5 時等同於 MAE；tau > 0.5 懲罰低估，tau < 0.5 懲罰高估
        errors = y_true - y_pred
        return torch.mean(torch.max(tau * errors, (tau - 1) * errors))
    
    def forward(self, y_pred, y_true):
        y_true = torch.log1p(y_true)
        loss= 1*self.MAE(y_pred, y_true[:, :])
        return loss

In [ ]:
def train_epoch(dataloader, model, criterion, optimizer, model_ema):
    scaler= amp.GradScaler()
    model.train()

    ep_loss= []
    for i, data in enumerate(tqdm(dataloader)):

        imgs= data['image'].to('cuda')
        labels= data['label'].to('cuda')
        
        with amp.autocast():
            preds= model(imgs)
            loss= criterion(preds, labels)
            ep_loss.append(loss.item())
            loss/= CFG['gradient_accumulation']
            scaler.scale(loss).backward()
            
            if (i+1) % CFG['gradient_accumulation']== 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            if model_ema: model_ema.update(model)
                
    return np.mean(ep_loss)

In [ ]:
from metrics import *
from sklearn.metrics import mean_absolute_error

def valid_epoch(dataloader, model, criterion):
    model.eval()
    
    ep_loss= []
    all_pred= []
    all_label= []
    for i, data in enumerate(tqdm(dataloader)):

        imgs= data['image'].to('cuda')
        labels= data['label'].to('cuda')
        all_label.extend(labels.cpu().numpy())
        
        with torch.no_grad():
            preds= model(imgs)
            loss= criterion(preds, labels)
            ep_loss.append(loss.item())
        all_pred.extend(preds.cpu().numpy())
        
    
    ## caculate metrics
    all_label= np.array(all_label)
    all_pred= np.array(all_pred)

    # Dry_Clover_g = all_pred[:, 2:3] - all_pred[:, 0:1]
    # Dry_Dead_g = all_pred[:, 1:2] - all_pred[:, 2:3]
    # Dry_Total_g = all_pred[:, 0:1] + all_pred[:, 1:2] + all_pred[:, 2:3]
    # GDM_g = all_pred[:, 0:1] + all_pred[:, 2:3]
    # all_pred = np.concatenate([Dry_Clover_g, Dry_Dead_g, all_pred], axis=1)

    mae = mean_absolute_error(all_label, all_pred)
    print(f'MAE: {mae}')
    r2 = weighted_r2_csiro(all_label, all_pred)
    print(f'Weighted R2: {r2}')
    
    score= r2
    return np.mean(ep_loss), score

# CFG

In [ ]:
[m for m in timm.list_models(pretrained=True) if 'v2_s' in m]

In [ ]:
CFG= {
    'fold': 0,
    'epoch': 100,
    'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
    
    'img_size': 896,
    'img_crop': None,
    
    'batch_size': 8,
    'gradient_accumulation': 1,
    'gradient_checkpoint': False,
    'drop_out': 0.3,
    'drop_path': 0.2,
    'mixup': 0.,
    'EMA': 0.995,
    
    'lr': 3e-4,
    'weight_decay': 0.,
    
    'num_classes': 5,
    'load_model':  False, #'./train_model/cv0_best.pth',
    'save_model': './train_model'
}

# Prepare Dataset

In [ ]:
ch_df = df= pd.read_csv('../Data/CH_DATA1.csv')[['image_path', 'kfold']]
for i in range(len(ch_df)): ch_df.loc[i, 'name'] = ch_df.loc[i, 'image_path'].split('/')[-1]
df= pd.read_csv('../Data/train_sgkf.csv')
for i in range(len(df)):
    name_ = df.loc[i, 'image_path'].split('/')[-1]
    fold = ch_df[ch_df['name']==name_]['kfold'].values[0]
    df.loc[i, 'fold'] = fold


In [ ]:
# df= pd.read_csv('../Data/train_sgkf.csv')
train_df= df[df['fold']!=CFG['fold']].reset_index(drop=True)
valid_df= df[df['fold']==CFG['fold']].reset_index(drop=True)
print(f'train dataset: {len(train_df)}')
print(f'valid dataset: {len(valid_df)}')

train_dataset= Customize_Dataset(train_df, get_train_transform(CFG['img_size']), training=True)
valid_dataset= Customize_Dataset(valid_df, get_test_transform(CFG['img_size']), training=False)

train_loader= DataLoader(train_dataset, batch_size= CFG['batch_size'], shuffle=True, num_workers=0)
valid_loader= DataLoader(valid_dataset, batch_size=1, shuffle=False, num_workers=0)
train_df.head()

In [ ]:
# import matplotlib.pyplot as plt
# cols = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

# plt.figure(figsize=(20,5))
# for i,c in enumerate(train_df.columns[-6:-1]):
#     plt.subplot(1,5,i+1)
#     plt.title(c)
#     train_df[c].hist(bins=50)
# plt.show()

In [ ]:
import matplotlib.pyplot as plt

data= train_dataset[3]
img= data['image']
label= data['label']
print(label)
plt.imshow(img.permute(1,2,0).numpy())
plt.show()

# Train

In [ ]:
## create model
if CFG['load_model']:
    print(f"load_model: {CFG['load_model']}")
    model= torch.load(CFG['load_model'], map_location= 'cuda')
else:
    # model= Customize_Model(CFG['model_name'], CFG['num_classes'])
    # model = huggingface_model(CFG['num_classes'])
    model= dinov3(CFG['num_classes'])
    
if CFG['gradient_checkpoint']: 
    print('use gradient checkpoint')
    model.model.set_grad_checkpointing(enable=True)
    
## EMA
model.to('cuda')
if CFG['EMA']:
    print(f"Use EMA: {CFG['EMA']}")
    model_ema= ModelEmaV2(model, decay=CFG['EMA'])
    model_ema.to('cuda')
else:
    model_ema= type('model_ema', (object,), {'module':{}})
    
## hyperparameter
for param in model.model.parameters():
    param.requires_grad = False

criterion= Customize_loss()
optimizer= optim.AdamW(model.single_head.parameters(), lr= CFG['lr'], weight_decay= CFG['weight_decay'])

## start training
best_score= -10000
train_losses= []
val_losses= []
val_score= []
for ep in range(1, CFG['epoch']+1):
    print(f'\nep: {ep}')
    
    if CFG['EMA']: train_loss= train_epoch(train_loader, model, criterion, optimizer, model_ema)
    else: 
        train_loss= train_epoch(train_loader, model, criterion, optimizer, False)
        model_ema.module= model
    
    if CFG['fold']==-1: 
        torch.save(model_ema.module, f"{CFG['save_model']}/cv{CFG['fold']}_ep{ep}.pth")
        continue

    valid_loss, valid_mae= valid_epoch(valid_loader, model_ema.module, criterion)
    print(f'train loss: {round(train_loss, 5)}')
    print(f'valid loss: {round(valid_loss, 5)}, valid_r2: {round(valid_mae, 5)}')
    
    train_losses.append(train_loss)
    val_losses.append(valid_loss)
    val_score.append(valid_mae)
    
    if valid_mae >= best_score:
        best_score= valid_mae
        torch.save(model_ema.module, f"{CFG['save_model']}/cv{CFG['fold']}_best.pth")
        print(f'model save at score: {round(best_score, 5)}')
        
    ## save model every epoch
    # torch.save(model_ema.module, f"{CFG['save_model']}/cv{CFG['fold']}_ep{ep}.pth")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,3))
plt.title('train/valid loss')
plt.plot(train_losses)
plt.plot(val_losses)
plt.show()

plt.figure(figsize=(5,3))
plt.title('val score')
plt.ylim(0, 1)
plt.plot(val_score)
plt.show()

# Model Soup

In [ ]:
import torch
import torch.nn as nn

class Customize_Model(nn.Module):
    def __init__(self, model_name, cls):
        super().__init__()
    def forward(self, image):
        x = self.model(image)
        x = self.linear(x)
        x = torch.expm1(x) if not self.training else x
        return x
    
def get_soup(state_dicts, alphal):
    sd = {k : state_dicts[0][k].clone() * alphal[0] for k in state_dicts[0].keys()}
    for i in range(1, len(state_dicts)):
        for k in state_dicts[i].keys():
            sd[k] = sd[k] + state_dicts[i][k].clone() * alphal[i]
    return sd


model_path=[
    f"{CFG['save_model']}/cv{CFG['fold']}_best.pth",
    # f"{CFG['save_model']}/cv{CFG['fold']}_ep{i}.pth" for i in range(90,91)
]
print(model_path)
model_weights= [torch.load(path, weights_only=False).state_dict() for path in model_path]
alphal= [1/len(model_path)]*len(model_path)

soup= get_soup(model_weights, alphal)
model_soup= torch.load(model_path[0], weights_only=False)
model_soup.load_state_dict(soup)

example = torch.rand(1, 3, CFG['img_size'], CFG['img_size']*2).cuda()
traced_script_module = torch.jit.trace(model_soup.cuda(), example)
traced_script_module.save(f"{CFG['save_model']}/cv{CFG['fold']}_soup.ts")

print('successful')

In [ ]:
# cv0: 0.872
# cv1: 0.873
# cv2: 0.835
# cv3: 0.841
# cv4: 0.831